In [ ]:
import socket

server_host = 'HYPE' # Or the IP address of HYPE
server_port = 1521

try:
    with socket.create_connection((server_host, server_port), timeout=5):
        print(f"Successfully connected to {server_host} on port {server_port}.")
except socket.error as e:
    print(f"Connection failed: {e}")
    print("Please check the SQL Server configuration and firewall settings on the server machine.")


In [2]:
import duckdb
import time

# 1. Connect to your local DuckDB file
con = duckdb.connect('c:\\duckDB\\adventuresales.duckdb')

# 2. Define the SQL query
sql_query = """
DROP TABLE IF EXISTS Complete_2years_Wk_Report_bySubChannel_All_Dtl_StockPoint;

CREATE TABLE Complete_2years_Wk_Report_bySubChannel_All_Dtl_StockPoint AS 
SELECT 
    f.region, a.branch_id, f.branch_name, a.year, a.month, a.week, 
    c.channel, c.sub_channel, 
    e.prodLine, e.prodgrp,
    d.udf6 AS psub_id, 
    g.depo, g.sp_name, g.sp_id, d.text_id, d.product, d.udf1 AS psub_id1,
    CASE WHEN c.CRM = 'Y' THEN 'Y' ELSE 'N' END AS CRM,
    CASE WHEN ((sls_id_chl <> '' AND y.sls_id <> '') OR sls_id_org <> '') AND y.no <= 100 THEN 'Y' ELSE 'N' END AS Sls_100,
    CASE WHEN (sls_id_chl <> '' AND y.sls_id <> '') OR sls_id_org <> '' THEN 'Y' ELSE 'N' END AS Sls_200,
    CASE WHEN u.custd_id <> '' THEN 'Y' ELSE 'N' END AS DW_18,
    CASE WHEN v.custd_id <> '' THEN 'Y' ELSE 'N' END AS DW_Oth,
    c.Retwhs,
    CASE WHEN c.Chiller = 'Y' THEN 'Y' ELSE 'N' END AS Chiller,
    CAST(SUM(a.net_qty) AS DECIMAL(17,5)) AS net_qty,
    CAST(SUM(a.net_amt) AS DECIMAL(17,5)) AS net_amt,
    CAST(SUM(a.grs_qty) AS DECIMAL(17,5)) AS grs_qty,
    CAST(SUM(a.grs_amt) AS DECIMAL(17,5)) AS grs_amt,
    CAST(SUM(a.rtn_qty) AS DECIMAL(17,5)) AS rtn_qty,
    CAST(SUM(a.rtn_amt) AS DECIMAL(17,5)) AS rtn_amt,
    CAST(SUM(a.grs_amt - a.net_amt - a.rtn_amt) AS DECIMAL(17,5)) AS disc_amt,
    CAST(SUM(i.disc_b_amt) AS DECIMAL(17,5)) AS Disc_B,
    CAST(SUM(j.disc_PotTunai) AS DECIMAL(17,5)) AS Disc_PT, 
    CASE WHEN x.sls_type LIKE '%canva%' THEN 'Y' ELSE 'N' END AS Canvass,
    CASE WHEN c.Mkt = 'Y' THEN 'Y' ELSE 'N' END AS Mkt
FROM sttwk AS a
LEFT OUTER JOIN customerdist_1910a c ON a.custd_id = c.kode_iap
LEFT OUTER JOIN product d ON a.prod_id = d.prod_id
LEFT OUTER JOIN prodcat e ON d.psub_id = e.psub_id
LEFT OUTER JOIN branch f ON a.branch_id = f.branch_id
LEFT OUTER JOIN stockpoint g ON a.sp_id = g.sp_id
LEFT OUTER JOIN discount_b i ON a.custd_id = i.custd_id AND a.prod_id = i.prod_id AND a.year = i.year AND a.week = i.week AND a.sls_id = i.sls_id
LEFT OUTER JOIN discount_potTunai j ON a.custd_id = j.custd_id AND a.prod_id = j.prod_id AND a.year = j.year AND a.week = j.week AND a.sls_id = j.sls_id
LEFT OUTER JOIN sls_chlprn h ON a.branch_id = h.br_chl AND a.sp_id = h.sp_chl AND a.sls_id = h.sls_id_chl AND a.year = h.year AND a.week = h.week
LEFT OUTER JOIN slsmanexc y ON h.br_chl = y.branch_id AND h.sp_chl = y.sp_id AND h.sls_id_prn = y.sls_id 
LEFT OUTER JOIN slsexc_chlprn6 b ON a.sp_id = b.sp_id1 AND a.sls_id = b.sls_id1 AND a.year = b.year AND a.week = b.week
LEFT OUTER JOIN dw_ilo_18 u ON a.custd_id = u.custd_id
LEFT OUTER JOIN dw_ilo_oth v ON a.custd_id = v.custd_id
LEFT OUTER JOIN slsman x ON a.branch_id = x.branch_id AND a.sp_id = x.sp_id AND a.sls_id = x.sls_id
GROUP BY ALL;
"""

start_time = time.time()

# 3. Execute the query via DuckDB
con.execute(sql_query)

end_time = time.time()
print("Report Table Created Successfully in DuckDB!")
print(f"Total time taken for write: {end_time - start_time:.2f} seconds")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Report Table Created Successfully in DuckDB!
Total time taken for write: 118.66 seconds
